In [1]:
import torch
import pandas as pd
import json
import os
from collections import defaultdict

# --- Configuration ---
# 1. 원본 VQA 테스트 데이터셋 CSV 파일 경로
#    이 파일은 'ID', 'img_path', 'Question', 'A', 'B', 'C', 'D' 컬럼을 포함해야 합니다.
VQA_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv" 
# 예: "./test_input.csv" 또는 "/data/your_challenge/test_input.csv"

# 2. 이전 단계에서 생성된 JSONL 파일 경로
#    이 파일은 각 이미지에 대해 A, B, C, D 보기가 'text_segment'로 포함된 파일입니다.
JSONL_FILE_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/coco_retrieval/coco_retrieval.test.jsonl"
# 예: "/data/your_challenge/coco_retrieval/coco_retrieval.test.jsonl"

# 3. BEiT-3 모델 평가 후 생성된 'score.json' 파일 경로
#    이 파일은 'scores', 'iids', 'tiids' 키를 포함해야 합니다.
SCORE_JSON_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/score.json"
# 예: "/data/your_challenge/model_outputs/score.json"

# 4. 예측 결과가 저장될 최종 출력 CSV 파일명
#    이 파일은 'ID', 'answer' 컬럼을 포함하게 됩니다.
PREDICTION_OUTPUT_CSV = "submission.csv" 

# --- End Configuration ---

# A, B, C, D 보기 선택을 위한 매핑
ANSWER_CHOICES = ["A", "B", "C", "D"]

def get_predictions_from_scores(
    vqa_csv_path: str, 
    jsonl_file_path: str, 
    score_json_path: str
) -> pd.DataFrame:
    """
    Retrieval model의 scores를 사용하여 VQA 객관식 답변을 예측하고, 
    ID,answer 형식의 pandas DataFrame을 반환합니다.
    """
    print(f"Loading VQA CSV from: {vqa_csv_path}")
    df_vqa = pd.read_csv(vqa_csv_path)

    print(f"Loading JSONL file from: {jsonl_file_path}")
    jsonl_data = []
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            jsonl_data.append(json.loads(line))

    print(f"Loading scores from: {score_json_path}")
    with open(score_json_path, 'r', encoding='utf-8') as f:
        # score.json이 행렬 데이터만 한 줄로 포함되어 있다고 가정
        # json.load()는 이 경우 리스트의 리스트(scores 행렬)를 직접 반환합니다.
        scores_list_of_lists = json.load(f) 
    
    scores = torch.tensor(scores_list_of_lists)

    # --- iids와 tiids 추론 (score.json 파일에 직접 포함되어 있지 않다고 가정) ---
    # scores 행렬의 차원을 기반으로 iids와 tiids를 추론합니다.
    # iids는 scores 행렬의 행 수 (고유 이미지 수)만큼 0부터 순차적으로 생성됩니다.
    iids = torch.arange(scores.shape[0]) 
    # tiids는 scores 행렬의 열 수 (총 텍스트 세그먼트 수)만큼 0부터 순차적으로 생성됩니다.
    tiids = torch.arange(scores.shape[1])

    print(f"Loaded scores shape: {scores.shape}")
    print(f"Inferred iids shape: {iids.shape}")
    print(f"Inferred tiids shape: {tiids.shape}")
    print("Warning: iids and tiids were inferred from score.json's dimensions. Ensure this matches BEiT-3's internal IDs.")
    # --- 추론 끝 ---

    # 예측 결과를 저장할 리스트
    predictions = []

    # JSONL의 `image_id` (0, 1, 2, ...)를 원본 VQA CSV의 'ID' (TEST_000, TEST_001, ...)로 매핑하기 위한 사전
    # 이 매핑은 `image_id`가 CSV 파일의 행 순서(0부터 시작)와 일치한다는 가정을 기반으로 합니다.
    jsonl_image_id_to_original_vqa_id = {
        i: df_vqa.loc[i, 'ID'] 
        for i in range(len(df_vqa)) 
    }

    # JSONL 파일의 `image_id` 별로 해당하는 텍스트 세그먼트의 글로벌 인덱스(`tiids`)를 그룹화
    # "이미지-보기 순서대로 4개씩" 가정에 따라 A, B, C, D 보기가 순서대로 추가되었다고 가정합니다.
    image_id_to_global_tiids = defaultdict(list)
    for global_tiid_idx, item in enumerate(jsonl_data):
        image_id_to_global_tiids[item["image_id"]].append(global_tiid_idx)

    # `iids` (scores 행렬의 행 인덱스에 해당하는 고유 이미지 ID)를 순회하며 각 이미지에 대한 예측 수행
    for iid_row_idx, current_image_id_from_iids_tensor in enumerate(iids):
        # 현재 처리 중인 이미지의 고유 ID (Python int 타입)
        current_image_id = current_image_id_from_iids_tensor.item() 

        # 현재 이미지 ID에 해당하는 원본 VQA의 'ID' 가져오기 (예: 'TEST_000')
        original_vqa_id = jsonl_image_id_to_original_vqa_id.get(current_image_id)
        if original_vqa_id is None:
            print(f"Warning: Could not map image_id {current_image_id} to original VQA ID. Skipping.")
            continue
            
        # `scores` 행렬에서 현재 이미지에 해당하는 모든 텍스트 세그먼트와의 유사도 점수를 가져옴
        image_scores_all_texts = scores[iid_row_idx, :] 
        
        # 현재 이미지의 4개 보기(A, B, C, D)에 해당하는 글로벌 텍스트 인덱스(`tiids`)를 가져옴
        # "이미지-보기 순서대로 4개씩" 가정에 따라 `sorted()`를 통해 순서 보장
        current_image_options_global_tiids = sorted(image_id_to_global_tiids[current_image_id])
        
        # 만약 어떤 이미지에 4개의 보기가 모두 존재하지 않는다면 경고 및 건너뛰기
        if len(current_image_options_global_tiids) != 4:
            print(f"Warning: Image ID {current_image_id} does not have exactly 4 options in JSONL. Skipping this image.")
            continue

        # 4개 보기 텍스트에 대한 유사도 점수만 추출
        option_scores = torch.tensor([image_scores_all_texts[tiid].item() for tiid in current_image_options_global_tiids])
        
        # 4개 보기 중 유사도 점수가 가장 높은 보기의 인덱스 (0, 1, 2, 3)를 찾음
        predicted_option_idx = torch.argmax(option_scores).item()
        
        # 찾은 인덱스를 'A', 'B', 'C', 'D'로 매핑
        predicted_answer_choice = ANSWER_CHOICES[predicted_option_idx]
        
        # 예측 결과를 리스트에 추가 (원하는 'ID', 'answer' 컬럼 형식으로)
        predictions.append({
            "ID": original_vqa_id,
            "answer": predicted_answer_choice 
        })
    # 예측 결과를 pandas DataFrame으로 변환하여 반환
    return pd.DataFrame(predictions)


In [9]:
# open json
with open(SCORE_JSON_PATH, 'r', encoding='utf-8') as f:
    scores = json.load(f)

# print(f'{len(scores)} scores loaded from {SCORE_JSON_PATH}')
# print(f'{len(scores[0])} scores loaded from {SCORE_JSON_PATH}')

# list 행렬 전환
scores = torch.tensor(scores)
sc = scores.T

# numpy로 변환
sc = sc.numpy()

scores = scores.numpy()

scores[2][8:12]

array([-0.00247977,  0.00017271,  0.00231925,  0.00537772], dtype=float32)

In [ ]:


try:
    # 예측 함수 호출
    prediction_df = get_predictions_from_scores(
        VQA_TEST_CSV_PATH, 
        JSONL_FILE_PATH, 
        SCORE_JSON_PATH
    )
    
    # 생성된 예측 DataFrame을 CSV 파일로 저장 (index=False는 행 번호를 제외)
    print(f"\nPredictions generated. Saving to {PREDICTION_OUTPUT_CSV}")
    prediction_df.to_csv(PREDICTION_OUTPUT_CSV, index=False)
    print("Done!")
    
    # 저장된 파일의 상위 5개 행을 출력하여 확인
    print(f"\nSample Predictions (from {PREDICTION_OUTPUT_CSV}):")
    print(prediction_df.head())

except FileNotFoundError as e:
    print(f"Error: Required file not found. {e}. Please check your file paths in the configuration section.")
except KeyError as e:
    print(f"Error: Missing expected key in input data (score.json or CSV). {e}. Please verify the file contents and their structure.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Loading VQA CSV from: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/data_2/test.csv
Loading JSONL file from: /data/2_data_server/cv-07/dice/2025_samsung_challenge/unilm/beit3/coco_retrieval/coco_retrieval.test.jsonl
Loading scores from: /data/2_data_server/cv-07/dice/2025_samsung_challenge/finetuning/unilm/beit3/score.json
An unexpected error occurred: list indices must be integers or slices, not str
